In [1]:
import os, glob
import numpy as np
from keras.preprocessing.image import load_img, img_to_array
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TensorBoard
from sklearn.model_selection import train_test_split

Using TensorFlow backend.


In [2]:
img_path = 'input/train'
img_ext = '.jpg'
img_form = img_path + '/{}' + img_ext

lbl_path = 'input/label'
lbl_ext = '.png'
lbl_form = lbl_path + '/{}' + lbl_ext

In [3]:
img_names = glob.glob(img_path + '/*' + img_ext)
img_names = [names.replace('\\', '/') for names in img_names]
img_names = [names.split('/')[-1].split('.')[0] for names in img_names]
print('Found {} images.'.format(len(img_names)))

Found 17 images.


In [4]:
from u_net import get_unet_128
input_size, model = get_unet_128()

W0823 01:12:34.339902  9116 deprecation_wrapper.py:119] From C:\Users\sein1\AppData\Local\Continuum\anaconda3\envs\tf\lib\site-packages\keras\backend\tensorflow_backend.py:74: The name tf.get_default_graph is deprecated. Please use tf.compat.v1.get_default_graph instead.

W0823 01:12:34.383879  9116 deprecation_wrapper.py:119] From C:\Users\sein1\AppData\Local\Continuum\anaconda3\envs\tf\lib\site-packages\keras\backend\tensorflow_backend.py:517: The name tf.placeholder is deprecated. Please use tf.compat.v1.placeholder instead.

W0823 01:12:34.390874  9116 deprecation_wrapper.py:119] From C:\Users\sein1\AppData\Local\Continuum\anaconda3\envs\tf\lib\site-packages\keras\backend\tensorflow_backend.py:4138: The name tf.random_uniform is deprecated. Please use tf.random.uniform instead.

W0823 01:12:34.462833  9116 deprecation_wrapper.py:119] From C:\Users\sein1\AppData\Local\Continuum\anaconda3\envs\tf\lib\site-packages\keras\backend\tensorflow_backend.py:174: The name tf.get_default_sessi

In [5]:
train, valid = train_test_split(img_names, test_size=0.10, random_state=42)

In [6]:
def train_generator():
    while True:
        for start in range(0, len(train)):
            x_batch = []
            y_batch = []
    
            id = train[start]
            img_path = img_form.format(id)
            mask_path = lbl_form.format(id)
            img = load_img(img_path,color_mode='rgb', target_size=(input_size,input_size))
            mask = load_img(mask_path, color_mode='grayscale', target_size=(input_size,input_size))
            img = img_to_array(img)
            mask = img_to_array(mask)
            x_batch.append(img)
            y_batch.append(mask)
            #print('x:{}, y:{}'.format(img.shape, mask.shape))
            
            x_batch = np.array(x_batch, np.float32) / 255
            y_batch = np.array(y_batch, np.float32) / 255
            yield x_batch, y_batch

In [7]:
def valid_generator():
    while True:
        for start in range(0, len(valid)):
            x_batch = []
            y_batch = []

            id = valid[start]    
            img_path = img_form.format(id)
            mask_path = lbl_form.format(id)
            img = load_img(img_path,color_mode='rgb', target_size=(input_size,input_size))
            mask = load_img(mask_path, color_mode='grayscale', target_size=(input_size,input_size))
            img = img_to_array(img)
            mask = img_to_array(mask)
            x_batch.append(img)
            y_batch.append(mask)
            x_batch = np.array(x_batch, np.float32) / 255
            y_batch = np.array(y_batch, np.float32) / 255
            yield x_batch, y_batch


In [8]:
callbacks = [
    EarlyStopping(
        monitor='val_dice_loss',
        patience=8,
        verbose=1,
        min_delta=1e-4,
        mode='max'),
    ReduceLROnPlateau(monitor='val_dice_loss',
                               factor=0.5,
                               patience=4,
                               verbose=1,
                               epsilon=1e-5,
                               mode='max'),
             ModelCheckpoint(monitor='val_dice_loss',
                             filepath='weights/best_weights.hdf5',
                             save_best_only=True,
                             save_weights_only=True,
                             mode='max'),
             TensorBoard(log_dir='logs')]

C:\Users\sein1\AppData\Local\Continuum\anaconda3\envs\tf\lib\site-packages\keras\callbacks.py:1065: UserWarning: `epsilon` argument is deprecated and will be removed, use `min_delta` instead.
  warnings.warn('`epsilon` argument is deprecated and '


In [ ]:
model.fit_generator(generator=train_generator(),
                    steps_per_epoch=15,
                    epochs=20,
                    verbose=2,
                    callbacks=callbacks,
                    validation_data=valid_generator(),
                    validation_steps=2)

W0823 01:12:45.331565  9116 deprecation_wrapper.py:119] From C:\Users\sein1\AppData\Local\Continuum\anaconda3\envs\tf\lib\site-packages\keras\callbacks.py:850: The name tf.summary.merge_all is deprecated. Please use tf.compat.v1.summary.merge_all instead.

W0823 01:12:45.331565  9116 deprecation_wrapper.py:119] From C:\Users\sein1\AppData\Local\Continuum\anaconda3\envs\tf\lib\site-packages\keras\callbacks.py:853: The name tf.summary.FileWriter is deprecated. Please use tf.compat.v1.summary.FileWriter instead.



Epoch 1/20
 - 27s - loss: 0.7829 - dice_loss: 0.5525 - val_loss: 0.6623 - val_dice_loss: 0.7395
Epoch 2/20
 - 25s - loss: 0.3911 - dice_loss: 0.7536 - val_loss: 7.6659 - val_dice_loss: 0.3524
Epoch 3/20
 - 26s - loss: 0.3022 - dice_loss: 0.8052 - val_loss: 0.3730 - val_dice_loss: 0.8206
Epoch 4/20
 - 25s - loss: 0.2539 - dice_loss: 0.8373 - val_loss: 0.4013 - val_dice_loss: 0.8262
Epoch 5/20
 - 32s - loss: 0.2400 - dice_loss: 0.8492 - val_loss: 1.0187 - val_dice_loss: 0.7195
Epoch 6/20
 - 29s - loss: 0.2428 - dice_loss: 0.8503 - val_loss: 0.4303 - val_dice_loss: 0.8417
Epoch 7/20
 - 29s - loss: 0.1878 - dice_loss: 0.8849 - val_loss: 0.2630 - val_dice_loss: 0.8855
Epoch 8/20
 - 26s - loss: 0.1472 - dice_loss: 0.9075 - val_loss: 0.1538 - val_dice_loss: 0.9169
Epoch 9/20
 - 26s - loss: 0.1273 - dice_loss: 0.9204 - val_loss: 0.3050 - val_dice_loss: 0.8851
Epoch 10/20
 - 25s - loss: 0.1481 - dice_loss: 0.9132 - val_loss: 2.6855 - val_dice_loss: 0.1023
Epoch 11/20
 - 26s - loss: 0.1301 - dic